In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import copy
from scipy.stats import wilcoxon, mannwhitneyu
import pickle as pkl

import nibabel as nib

import matplotlib.pyplot as plt 

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

In [ ]:
def load_unet_result(path, _print):
    unet_df = pd.read_csv(path, index_col = 'Unnamed: 0')
    index_values = []
    for _index in unet_df.index:
        index_values.append(_index.split('-seg')[0]  # strip '-seg' suffix added by U-Net inference script)

    unet_df.index = index_values  
    unet_df.drop(['WT jaccard', 'TC jaccard', 'ET jaccard']  # Dice is the primary metric; Jaccard unused, axis = 1, inplace = True)
    summary_unet_df = pd.DataFrame(zip(unet_df.mean().values.tolist(), 
                                       unet_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = unet_df.columns)
    if _print:
        print("****UNet******")
        print(summary_unet_df)
    return summary_unet_df, unet_df

def load_nnunet_result(path, _print):
    with open(path, 'r') as file:
        data = json.load(file)

    WT = []
    TC = []
    ET = []
    file_name = []
    for case in data['metric_per_case']:
        WT.append(case['metrics']['(2, 1, 3)']['Dice'])
        TC.append(case['metrics']['(2, 3)']['Dice'])
        ET.append(case['metrics']['(3,)']['Dice'])
        file_name.append(case['reference_file'].split('/')[-1].split('.')[0])

    nnunet_df = pd.DataFrame(zip(WT, TC, ET), columns = ['WT dice', 'TC dice', 'ET dice'], 
                             index = file_name)
    summary_nnunet_df = pd.DataFrame(zip(nnunet_df.mean().values.tolist(), 
                                       nnunet_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = nnunet_df.columns)
    if _print:
        print("****nnUNet******")
        print(summary_nnunet_df)
    return summary_nnunet_df, nnunet_df

def load_TransBTS_result(path, _print):
    with open(path, 'r') as file:
        data = json.load(file)

    WT = []
    TC = []
    ET = []
    file_name = []
    for case_id in data.keys():
        case = data[case_id]
        WT.append(case['WT'][0])
        TC.append(case['TC'][0])
        ET.append(case['ET'][0])
        file_name.append(case_id)

    TransBTS_df = pd.DataFrame(zip(WT, TC, ET), columns = ['WT dice', 'TC dice', 'ET dice'], 
                             index = file_name)
    summary_TransBTS_df = pd.DataFrame(zip(TransBTS_df.mean().values.tolist(), 
                                       TransBTS_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = TransBTS_df.columns)
    if _print:
        print("****TransBTS******")
        print(summary_TransBTS_df)
    return summary_TransBTS_df, TransBTS_df

In [ ]:
def read_results(_print=True):
    path = '../Results/Result/Vanilla_Unet/Unet_test_dice.csv'
    summary_unet_df, unet_df = load_unet_result(path, _print)

    path = '../Results/Result/nnUnet/nnUNetTrainer/summary.json'
    summary_da_nnunet_df, nnunet_da_df = load_nnunet_result(path, _print)

    path = '../Results/Result/nnUnet/nnUNetTrainerNoDA/summary.json'
    summary_noda_nnunet_df, nnunet_noda_df = load_nnunet_result(path, _print)

    path = '../Results/Result/TransBTS/submission/TransBTS2023-11-03/TransBTS_summary.json'
    summary_TransBTS_df, TransBTS_df = load_TransBTS_result(path, _print)
    return unet_df, nnunet_noda_df, nnunet_da_df, TransBTS_df




In [ ]:
def get_overlaps(unet_df, TransBTS_df, nnunet_noda_df, WT_dice_threshold, TC_dice_threshold, ET_dice_threshold):
    unet_df_sub = unet_df[(unet_df['WT dice'] < WT_dice_threshold) 
                          & (unet_df['TC dice'] < TC_dice_threshold) 
                          & (unet_df['ET dice'] < ET_dice_threshold)]
    
    TransBTS_df_sub = TransBTS_df[(TransBTS_df['WT dice'] < WT_dice_threshold) 
                          & (TransBTS_df['TC dice'] < TC_dice_threshold) 
                          & (TransBTS_df['ET dice'] < ET_dice_threshold)]
    
    nnunet_noda_df_sub = nnunet_noda_df[(nnunet_noda_df['WT dice'] < WT_dice_threshold) 
                          & (nnunet_noda_df['TC dice'] < TC_dice_threshold) 
                          & (nnunet_noda_df['ET dice'] < ET_dice_threshold)]

    unet_df_sub_subjects = unet_df_sub.index.values.tolist()
    TransBTS_df_sub_subjects = TransBTS_df_sub.index.values.tolist()
    nnunet_noda_df_sub_subjects = nnunet_noda_df_sub.index.values.tolist()

    all_overlaps = list(set(unet_df_sub_subjects) & set(TransBTS_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
    print('all overlap', len(all_overlaps), unet_df_sub.shape, nnunet_noda_df_sub.shape)

    unet_nnunet_overlaps = list(set(unet_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
    print('unet-nnunet overlap', len(unet_nnunet_overlaps))

    unet_TransBTS_overlaps = list(set(unet_df_sub_subjects) & set(TransBTS_df_sub_subjects))
    print('unet-TransBTS overlap', len(unet_TransBTS_overlaps))

    nnunet_TransBTS_overlaps = list(set(TransBTS_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
    print('nnunet-TransBTS overlap', len(nnunet_TransBTS_overlaps))
    
    return all_overlaps, unet_nnunet_overlaps


In [ ]:
def read_radiomics_results(analysis_type):
    file_name = '../Results/Analysis_Results/Radiomics/Tumor/' + analysis_type + '.pkl'
    with open(file_name, 'rb') as f:
        results = pkl.load(f)
    return results

def read_MRI(dataset, patient_id):
    if dataset == 'Brats2020':
        baseloc = '../input/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/'
        pefix = 'BraTS20_Training_' + patient_id + '/' + 'BraTS20_Training_' + patient_id
        suffixs = ['_flair.nii','_t2.nii', '_t1.nii', '_t1ce.nii', '_seg.nii']
    elif dataset == 'Brats2023':
        baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
        pefix = patient_id + '/' + patient_id
        suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz']

    flair_filename = baseloc + pefix + suffixs[0]
    flair_img_f = nib.load(flair_filename)
    flair_img = np.asarray(flair_img_f.dataobj)

    t2_filename = baseloc + pefix + suffixs[1]
    t2_img_f = nib.load(t2_filename)
    t2_img = np.asarray(t2_img_f.dataobj)

    t1_filename = baseloc + pefix + suffixs[2]
    t1_img_f = nib.load(t1_filename)
    t1_img = np.asarray(t1_img_f.dataobj)

    t1ce_filename = baseloc + pefix + suffixs[3]
    t1ce_img_f = nib.load(t1ce_filename)
    t1ce_img = np.asarray(t1ce_img_f.dataobj)
    
    mask_filename = baseloc + pefix + suffixs[4]
    mask_img_f = nib.load(mask_filename)
    mask_img = np.asarray(mask_img_f.dataobj)
    
    return flair_img, t2_img, t1_img, t1ce_img, mask_img 

def preprocess_mask_labels(mask):
    # whole tumour
    mask_WT = mask.copy()
    mask_WT[mask_WT == 1] = 1
    mask_WT[mask_WT == 2] = 1
    mask_WT[mask_WT == 3] = 1
    # include all tumours 

    # NCR / NET - LABEL 1
    mask_TC = mask.copy()
    mask_TC[mask_TC == 1] = 1
    mask_TC[mask_TC == 2] = 0
    mask_TC[mask_TC == 3] = 1
    # exclude 2 / 4 labelled tumour 

    # ET - LABEL 4 
    mask_ET = mask.copy()
    mask_ET[mask_ET == 1] = 0
    mask_ET[mask_ET == 2] = 0
    mask_ET[mask_ET == 3] = 1
    # exclude 2 / 1 labelled tumour 

    mask = np.stack([mask_WT, mask_TC, mask_ET])
    
    return mask 

def read_mask_file(patient_id):
    baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
    pefix = patient_id + '/' + patient_id
    suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz']
    outputloc = '../Results/Result/Vanilla_Unet/' + patient_id
    
    sample_filename_mask = baseloc + pefix + suffixs[4]
    sample_mask_f = nib.load(sample_filename_mask)
    sample_mask = np.asarray(sample_mask_f.dataobj)
    
    masks = preprocess_mask_labels(sample_mask)
    mask_WT, mask_TC, mask_ET = masks[0], masks[1], masks[2]
    
    return mask_WT

def remove_duplicate_columns(df):
    """
    Remove columns from a DataFrame that have identical values to other columns, regardless of the column name.
    
    Parameters:
    - df: The pandas DataFrame from which to remove duplicate columns.
    
    Returns:
    - A new DataFrame with duplicate columns removed.
    """
    columns_to_remove = set()
    for i in range(df.shape[1]): # Iterate over all columns
        for j in range(i + 1, df.shape[1]): # Compare each column with every other column
            # Check if not already identified as duplicate and if equal
            if i not in columns_to_remove and j not in columns_to_remove:
                if df.iloc[:, i].equals(df.iloc[:, j]):
                    columns_to_remove.add(j)
    
    # Create a new DataFrame without the duplicate columns
    df_cleaned = df.drop(columns=df.columns[list(columns_to_remove)])
    return df_cleaned

In [ ]:
# Thresholds from paper: cases below all three are concordant-poor
WT_dice_threshold= 0.91
TC_dice_threshold= 0.86
ET_dice_threshold= 0.85


analysis_types = ['firstorder', 'glcm_1', 
                  'glcm_5', 
                  'glcm_10', 
                  'gldm_1', 'gldm_5',
                  'gldm_10', 
                  'glrlm', 
                  'glszm', 'intensity', 
                  'ngtdm_1', 'ngtdm_5',
                  'ngtdm_10',
                  'shape', 'size' ]

unet_df, nnunet_noda_df, nnunet_da_df, TransBTS_df = read_results(False)

all_overlaps, unet_nnunet_overlaps = get_overlaps(unet_df, 
                                                TransBTS_df, 
                                                nnunet_noda_df, 
                                                WT_dice_threshold, 
                                                TC_dice_threshold, 
                                                ET_dice_threshold)

In [ ]:
results_df = unet_df
for analysis_type in analysis_types:
    try:
        radiomics_results_df = read_radiomics_results(analysis_type)
    except:
        continue
    properties = {}
    property_df = pd.DataFrame()
    for i in range(len(radiomics_results_df.keys())):
        key = list(radiomics_results_df.keys())[i]
        properties[i] = key
#         print('properties ', i, ':', key)

    for i in range(len(properties)):
        selected_property = properties[i]

        property_result_df = pd.DataFrame.from_dict(radiomics_results_df[selected_property], 
                                                    orient = 'index').astype(float)
        new_col = []
        for col in property_result_df.columns:
            new_col.append(selected_property + '_' + col + '_' + analysis_type)
        property_result_df.columns = new_col
        results_df = pd.merge(property_result_df, 
                       results_df, 
                       left_index=True, 
                       right_index=True)

In [ ]:
unet_df_bad = results_df[(results_df['WT dice'] < WT_dice_threshold) 
                          & (results_df['TC dice'] < TC_dice_threshold) 
                          & (results_df['ET dice'] < ET_dice_threshold)]
unet_df_bad = unet_df_bad[unet_df_bad.index.isin(unet_nnunet_overlaps)]
unet_df_bad['dice'] = ['bad']*unet_df_bad.shape[0]

unet_df_good = results_df[(results_df['WT dice'] >= WT_dice_threshold) 
                          & (results_df['TC dice'] >= TC_dice_threshold) 
                          & (results_df['ET dice'] >= ET_dice_threshold)]
unet_df_good['dice'] = ['good']*unet_df_good.shape[0]

prediction_df = pd.concat([unet_df_good, unet_df_bad])
prediction_df.shape

remaining_df = results_df[~results_df.index.isin(prediction_df.index)]
remaining_df['dice'] = ['bad']*remaining_df.shape[0]

# prediction_df = unet_df_good

# prediction_df = pd.concat([prediction_df, remaining_df])

In [ ]:
summary_df = pd.read_csv('../Results/Analysis_Results/Radiomics/summary/Tumor_summary.csv')
summary_df['property'] = summary_df['property'] + '_' +summary_df['Unnamed: 0'] + '_' +summary_df['analysis_type']

# important_features = summary_df[(summary_df['Statistically_Different'] == '***') 
#                                 & ((summary_df['Effect_Size'] == 'Large') 
#                                    | (summary_df['Effect_Size'] == 'Medium') 
#                                    | (summary_df['Effect_Size'] == 'Small'))].property.values.tolist()

important_features = summary_df[(summary_df['Statistically_Different'] == '***') 
                                & ((summary_df['Effect_Size'] == 'Large'))].property.values.tolist()

important_features.append('dice')
print(len(important_features))

for feature in prediction_df.columns: 
    if feature not in important_features:
        prediction_df = prediction_df.drop(feature, axis = 1)
        
for feature in remaining_df.columns: 
    if feature not in important_features:
        remaining_df = remaining_df.drop(feature, axis = 1)


prediction_df.shape, remaining_df.shape

In [ ]:
# correlation_matrix = prediction_df.corr().abs()
# threshold = 0.9

# # Identify pairs of highly correlated features
# highly_correlated_pairs = [(i, j) for i in range(correlation_matrix.shape[0]) for j in range(i+1, correlation_matrix.shape[1]) if correlation_matrix.iloc[i, j] > threshold]

# # Extract the unique feature names to be dropped
# features_to_drop = set([correlation_matrix.columns[j] for i, j in highly_correlated_pairs])
# len(features_to_drop)
# prediction_df = prediction_df.drop(columns=features_to_drop)

In [ ]:
prediction_df = remove_duplicate_columns(prediction_df)

prediction_df.reset_index(inplace=True, drop=True)

for feature in remaining_df.columns: 
    if feature not in prediction_df.columns:
        remaining_df = remaining_df.drop(feature, axis = 1)

prediction_df.shape, remaining_df.shape

In [ ]:
# Separate features and target variable
X = prediction_df.drop('dice', axis=1)  # Feature matrix
y = prediction_df['dice']  # Target variable

In [ ]:
perf_score = [] 
conf_matrices = []

for _rand in [43,42,87,12,176]:
    # Initialize the Stratified K-Fold cross-validator
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=_rand)

    # Lists to store results of each fold
    accuracies = []
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns = X.columns)

    # Stratified K-Fold Cross-Validation
    for train_index, test_index in skf.split(X_scaled, y):
        # Split data
        X_train, X_test = X_scaled.loc[train_index], X_scaled.loc[test_index]
        y_train, y_test = y.loc[train_index], y.loc[test_index]
        
        sm = SMOTE(sampling_strategy = 'not majority', 
                   k_neighbors=5, 
                   random_state=42)
        X_res, y_res = sm.fit_resample(X_train, y_train)

        # Initialize and train the classifier
#         clf = RandomForestClassifier(n_estimators = 100, 
#                                      criterion = 'gini', 
#                                      class_weight= 'balanced', 
#                                      max_depth=10, 
#                                      min_samples_split=20)
        clf = LogisticRegression(random_state=42, 
                                 penalty = 'elasticnet', 
                                 solver = 'saga', 
                                 l1_ratio=0.5, 
                                 max_iter=5000)
#         clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
       
        clf.fit(X_train, y_train)
#         clf.fit(X_res, y_res)

        # Predict on the test set
        y_pred = clf.predict(X_test)

        # Evaluate the classifier
        acc = accuracy_score(y_test, y_pred)
        accuracies.append(acc)
        
        conf_matrices.append(confusion_matrix(y_test, y_pred))
        
        print(classification_report(y_test, y_pred))
        
        f1 = f1_score(y_test, y_pred, average=None)
        perf_score.append(f1)
        

    # Display the average accuracy across all folds
    print(f'Average Accuracy: {np.mean(perf_score[:]):.2f}')

    # You can also analyze the confusion matrices to understand the model's performance in more detail.

In [ ]:
perf_score_df = pd.DataFrame(perf_score, columns = ['bad', 'good'])
perf_score_df.mean()